# Simple student workflow: Plan, Evidence, Dataset

This notebook rehearses the public Python API with a scripted substitute for your coding agent. All text is synthetic. There are no network calls, credentials, or paid providers. It verifies workflow mechanics, not meaning accuracy or recall.

For a real study, use the packaged V6 skill or MCP tools to have your own agent prepare a plan within your permitted sources and budget. The browser shows Plan, Evidence, and Dataset. You do not need to maintain the component JSON files yourself.


In [ ]:
from copy import deepcopy
import json
from pathlib import Path
import tempfile
from quarry import Project, FakeProvider, Response
from quarry.config import experts
from quarry.workflow import prepare_review, approve_review, run_review, release_dataset

workspace = tempfile.TemporaryDirectory(prefix="v6-student-rehearsal-")
base = Path(workspace.name)
corpus = base / "corpus"
corpus.mkdir()
text = "😀 Café is a fictional place here. Our community calls a brief pause moonstep."
source = corpus / "synthetic.md"
source.write_text(text, encoding="utf-8")
project = Project(base / "my-study")


## Plan

The researcher describes the question and limits. In normal use the coding agent prepares the bundle; the next cell is an explicit offline surrogate. The prepared review captures sources, freezes blank whole-source annotation packets, and estimates parsed-length costs before any extraction job.


In [ ]:
setup = project.prepare_agent_setup({
    "research_question": "Which community expressions and senses occur in the permitted sources?",
    "data_type": "term", "local_roots": [str(corpus)], "languages": ["en"],
    "communities": ["synthetic-guide"], "mode": "broad", "model_profile": "fake",
    "budget_usd": 0, "max_sources": 1,
})
agent_bundle = deepcopy(setup["starter_bundle"])
agent_bundle["unresolved_questions"] = []
agent_bundle["agent"].update(tool="scripted_notebook_surrogate", model="not_a_real_agent")
agent_bundle["recipe"]["execution"]["concurrency"] = 1
agent_bundle["sources"] = [{
    "source_key": "fictional-1", "kind": "local", "locator": str(source),
    "metadata": {"language": "en", "community": "synthetic-guide", "domain": "synthetic",
        "publication_date": None, "publication_date_reason": "fictional_undated",
        "constraints": {"content_origin": "synthetic_fixture", "license": "test-only",
            "permitted_use": "offline_rehearsal", "redistribution": "synthetic_only"}},
    "why_selected": "Synthetic workflow rehearsal", "origin": "researcher_supplied",
    "search_refs": [], "family_id": "fictional-family",
}]
review = prepare_review(project, agent_bundle, fetch_remote=False)
assert project.store.count("jobs") == 0
{k: review[k] for k in ("question", "mode", "model_profile", "sources", "estimate", "rigor_note")}


Read the review before deciding. This cell records a synthetic approval and runs fake responses. In a real study your actual approval comes first; approval declarations do not prove a person's identity. A real provider also requires explicit paid authorization. Human gold must come from independent annotation of complete frozen sources, not this scripted proposal.


In [ ]:
approve_review(project, review["review_id"], approval_origin="notebook",
               approval_reference="Synthetic rehearsal decision")
candidate = {"expression": "moonstep", "language": "en", "sense": "A brief community pause",
    "context": text, "start": None, "definition_quote": None, "translation": None, "attributes": {}}
replies = [Response(json.dumps({"candidates": [candidate] if n == 0 else [],
    "coverage_complete": True}, ensure_ascii=False), 40, 40, "stop", "synthetic-notebook-v1")
    for n in range(len(experts(project.recipe, project.profile)))]
provider = FakeProvider([], replies)
partial = run_review(project, review["review_id"], provider=provider, max_jobs=1)
assert partial["jobs"].get("pending", 0) > 0
project = Project(project.path)  # another process can reopen the same project
resumed = run_review(project, review["review_id"], provider=provider)
assert len(provider.calls) == len(replies)
resumed["review_scope"]


## Evidence

The single-model discovery remains in the complete inventory. The exact slice uses Unicode code-point offsets, including the emoji and decomposed accent preceding it. The sense remains a proposal. Annotation and undo retain history.


In [ ]:
inventory = project.inventory()
record = inventory["records"][0]
occurrence = inventory["occurrences"][0]
assert text[occurrence["start"]:occurrence["end"]] == "moonstep"
annotation_id = project.annotate(record["record_id"], "approved", note="Synthetic human rehearsal")
project.undo_annotation(annotation_id)
assert project.inventory()["records"][0]["review_status"] == "unreviewed"
{"record": record, "evidence": occurrence, "annotation_history": project.inventory()["annotations"]}


## Dataset

A named release freezes records, runs the publication audit, and exports CSV, JSONL, Parquet, and annotation files. The audit establishes internal consistency, not truth, recall, rights, or publication acceptance. Reusing the release reproduces its saved records even after later live annotations.


In [ ]:
release = release_dataset(project, "student-rehearsal-v1")
export_dir = Path(release["out"])
exported_bytes = {p.name: p.read_bytes() for p in export_dir.iterdir() if p.is_file()}
project.annotate(record["record_id"], "needs_review", "Later live annotation")
repeat = release_dataset(project, "student-rehearsal-v1")
assert repeat["version_id"] == release["version_id"]
assert exported_bytes == {p.name: p.read_bytes() for p in export_dir.iterdir() if p.is_file()}
assert release["audit"]["integrity_passed"]
{"version_id": release["version_id"], "export_dir": str(export_dir), "files": release["files"],
 "issues": release["audit"]["issues"], "limitations": release["audit"]["limitations"]}


The temporary workspace exists for this kernel session. Copy desired synthetic artifacts before cleaning it up. Real studies should use a protected persistent directory.

See `docs/SIMPLE_WORKFLOW.md` for publication-inspired student examples, `docs/AGENT_NATIVE.md` for the skill/MCP setup, and `docs/EVALUATION.md` for independent complete-source gold and grouped held-out evaluation. An unchanged captured plan resumes after original files move. After jobs exist, a changed plan cannot be claimed as a new blind corpus; use an explicit advanced protocol or separate study.
